In [1]:
#Load Libraries
library(Seurat)
library(SeuratDisk)
library(SeuratWrappers)
library(Signac)
library(reticulate)
library(sceasy)
library(Matrix)
library(future)

#Set Options
options(future.globals.maxSize = 300000 * 1024^2) #for 300GB max size
plan("multicore", workers = 4)

#Set working directory
setwd("/storage1/fs1/jmillman/Active/DigitalTwin")

Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built with package ‘Matrix’ 1.7.3 but the current
version is 1.7.4; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat



# Load Data

In [2]:
# load RNA data
data.rna <- readRDS("checkpoints/DT_RNAintegrated.rds")
Idents(data.rna) <- 'dataset'
DefaultAssay(object = data.rna) <- "RNA"
data.rna

An object of class Seurat 
35462 features across 333580 samples within 1 assay 
Active assay: RNA (35462 features, 2000 variable features)
 3 layers present: data, counts, scale.data
 4 dimensional reductions calculated: pca, umap.merged.rna, harmony, umap.harmony

In [3]:
colnames(data.rna[[]])

[1] "orig.ident"       "nCount_RNA"       "nFeature_RNA"     "modality"        
 [5] "dataset"          "protocol"         "stage"            "day"             
 [9] "cellsource"       "gender"           "pct_mito"         "harmony_clusters"

In [4]:
# load ATAC data
data.atac <- readRDS("checkpoints/DT_ATACintegrated.rds")
Idents(data.atac) <- 'dataset'
DefaultAssay(object = data.atac) <- "ATAC"
data.atac

An object of class Seurat 
569068 features across 209052 samples within 1 assay 
Active assay: ATAC (569068 features, 0 variable features)
 2 layers present: counts, data
 2 dimensional reductions calculated: integrated_lsi, umap.integrated.atac

In [5]:
colnames(data.atac[[]])

[1] "orig.ident"                        "nCount_ATAC"                      
 [3] "nFeature_ATAC"                     "gex_barcode"                      
 [5] "atac_barcode"                      "is_cell"                          
 [7] "excluded_reason"                   "gex_raw_reads"                    
 [9] "gex_mapped_reads"                  "gex_conf_intergenic_reads"        
[11] "gex_conf_exonic_reads"             "gex_conf_intronic_reads"          
[13] "gex_conf_exonic_unique_reads"      "gex_conf_exonic_antisense_reads"  
[15] "gex_conf_exonic_dup_reads"         "gex_exonic_umis"                  
[17] "gex_conf_intronic_unique_reads"    "gex_conf_intronic_antisense_reads"
[19] "gex_conf_intronic_dup_reads"       "gex_intronic_umis"                
[21] "gex_conf_txomic_unique_reads"      "gex_umis_count"                   
[23] "gex_genes_count"                   "atac_raw_reads"                   
[25] "atac_unmapped_reads"               "atac_lowmapq"                     
[27] "atac_dup_reads"                    "atac_chimeric_reads"              
[29] "atac_mitochondrial_reads"          "atac_fragments"                   
[31] "atac_TSS_fragments"                "atac_peak_region_fragments"       
[33] "atac_peak_region_cutsites"         "nucleosome_signal"                
[35] "nucleosome_percentile"             "TSS.enrichment"                   
[37] "TSS.percentile"                    "modality"                         
[39] "dataset"                           "protocol"                         
[41] "stage"                             "day"                              
[43] "cellsource"                        "gender"                           
[45] "total"                             "duplicate"                        
[47] "chimeric"                          "unmapped"                         
[49] "lowmapq"                           "mitochondrial"                    
[51] "nonprimary"                        "passed_filters"                   
[53] "is__cell_barcode"                  "TSS_fragments"                    
[55] "DNase_sensitive_region_fragments"  "enhancer_region_fragments"        
[57] "promoter_region_fragments"         "on_target_fragments"              
[59] "blacklist_region_fragments"        "peak_region_fragments"            
[61] "peak_region_cutsites"

# Preprocess Data

In [6]:
# convert to v3 assay
DefaultAssay(data.rna) <- 'RNA'
data.rna[["RNA"]] <- as(object = data.rna[["RNA"]], Class = "Assay")

Warning message:
“Assay RNA changing from Assay5 to Assay”


In [7]:
# Remove extra metadata
columns.to.keep <- c("modality","dataset","protocol","stage","day","cellsource","gender")

for(i in colnames(data.rna[[]])) {
    if(i %in% columns.to.keep){
        print(i)
    } else{
        data.rna[[i]] <- NULL
    } 
}

for(i in colnames(data.atac[[]])) {
    if(i %in% columns.to.keep){
        print(i)
    } else{
        data.atac[[i]] <- NULL
    } 
}

[1] "modality"
[1] "dataset"
[1] "protocol"
[1] "stage"
[1] "day"
[1] "cellsource"
[1] "gender"
[1] "modality"
[1] "dataset"
[1] "protocol"
[1] "stage"
[1] "day"
[1] "cellsource"
[1] "gender"


# Export as AnnData (.h5ad)

In [8]:
py_require("anndata")
anndata <- import("anndata")

In [9]:
# convert RNA data to  anndata
convertFormat(data.rna, from="seurat", to="anndata", main_layer="counts", assay="RNA", drop_single_values=FALSE, outFile='checkpoints/rna.h5ad')
rm(data.rna)

Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the sceasy package.
  Please report the issue to the authors.”


AnnData object with n_obs × n_vars = 333580 × 35462
    obs: 'modality', 'dataset', 'protocol', 'stage', 'day', 'cellsource', 'gender'
    var: 'vf_vst_counts.Balboa_S5DT_mean', 'vf_vst_counts.Balboa_S5DT_variance', 'vf_vst_counts.Balboa_S5DT_variance.expected', 'vf_vst_counts.Balboa_S5DT_variance.standardized', 'vf_vst_counts.Balboa_S5DT_variable', 'vf_vst_counts.Balboa_S5DT_rank', 'vf_vst_counts.Balboa_S5NT_mean', 'vf_vst_counts.Balboa_S5NT_variance', 'vf_vst_counts.Balboa_S5NT_variance.expected', 'vf_vst_counts.Balboa_S5NT_variance.standardized', 'vf_vst_counts.Balboa_S5NT_variable', 'vf_vst_counts.Balboa_S5NT_rank', 'vf_vst_counts.Balboa_S7w0_A11_mean', 'vf_vst_counts.Balboa_S7w0_A11_variance', 'vf_vst_counts.Balboa_S7w0_A11_variance.expected', 'vf_vst_counts.Balboa_S7w0_A11_variance.standardized', 'vf_vst_counts.Balboa_S7w0_A11_variable', 'vf_vst_counts.Balboa_S7w0_A11_rank', 'vf_vst_counts.Balboa_S7w0_B3_mean', 'vf_vst_counts.Balboa_S7w0_B3_variance', 'vf_vst_counts.Balboa_S7w0_B

In [10]:
# convert ATAC data to anndata
convertFormat(data.atac, from="seurat", to="anndata", main_layer="counts", assay="ATAC", drop_single_values=FALSE, outFile='checkpoints/atac.h5ad')

AnnData object with n_obs × n_vars = 209052 × 569068
    obs: 'modality', 'dataset', 'protocol', 'stage', 'day', 'cellsource', 'gender'
    var: 'name'
    obsm: 'X_integrated_lsi', 'X_umap.integrated.atac'

In [11]:
sink("Notebooks/3_DigitalTwin/05_DT_DataConversion-sessionInfo.txt")
sessionInfo()
sink()

R version 4.5.1 (2025-06-13)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/atlas/libblas.so.3.10.3 
LAPACK: /usr/lib/x86_64-linux-gnu/atlas/liblapack.so.3.10.3;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] future_1.67.0         Matrix_1.7-4          sceasy_0.0.7         
 [4] reticulate_1.44.0     Signac_1.16.0         SeuratWrappers_0.4.0 
 [7] SeuratDisk_0.0.0.9021 Seurat_5.3.1.9999     SeuratObject_5.2.0   
[10] sp_2.2-0             

loaded via a name